<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/ViT-Experiments/ViT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q transformers torch

In [19]:
!pip install -q timm

In [20]:
!pip install -q evaluate

In [21]:
import os

# Set to True to always fine-tune, False to load existing checkpoints if available
FORCE_FINE_TUNING = False

In [22]:
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Using a smaller ViT model for reduced RAM usage
model_id = "vit_large_patch16_rope_mixed_ape_224"

model = timm.create_model(model_id, pretrained=True)
model.eval()

config = timm.data.resolve_model_data_config(model)
print(config)
vit_transform = timm.data.create_transform(**config)
print(vit_transform)

# Load an AutoModelForImageClassification from Hugging Face Transformers to get ImageNet-1k id2label mapping
# This will be used for both ViT and ResNet predictions for consistency.
id2label_model = AutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224")
id2label_mapping = id2label_model.config.id2label

print(f"Loaded ViT model: {model_id}")

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}
Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loaded ViT model: vit_large_patch16_rope_mixed_ape_224


In [23]:
from datasets import load_dataset

# Load the full training split first
food_full_train = load_dataset("ethz/food101", split="train")

# Shuffle the full training split and select the first 5000 examples
# This ensures a more diverse set of classes in the subset
food = food_full_train.shuffle(seed=42).select(range(5000))

# Now perform the train-test split on this diverse subset
food = food.train_test_split(test_size=0.2, shuffle=True, seed=42)

In [24]:
labels = food["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [25]:
from transformers import AutoImageProcessor

checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [26]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])

In [27]:
def transforms(examples):
    # Check if 'image' key exists. If not, it means the batch has already been transformed
    # (e.g., in a previous epoch or due to internal dataset handling).
    if "image" in examples:
        examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
        del examples["image"]
    return examples

food = food.with_transform(transforms)

In [28]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [29]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Update `compute_metrics` Function

### Subtask:
Modify the `compute_metrics` function to include the F1-score in addition to accuracy. This will provide a more comprehensive evaluation metric, especially for imbalanced datasets.

In [30]:
import evaluate
import numpy as np

# Load the F1 metric
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    # eval_pred is a PredictionOutput object
    predictions_logits = eval_pred.predictions
    labels = eval_pred.label_ids

    predictions = np.argmax(predictions_logits, axis=1)

    # Compute accuracy
    accuracy_result = accuracy.compute(predictions=predictions, references=labels)

    # Compute F1-score. Use 'weighted' average for multi-class classification.
    f1_result = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    # Compute Top-5 Accuracy
    # For top-k accuracy, we need to sort the logits and check if the true label is among the top-k
    k = 5
    top_k_predictions = np.argsort(predictions_logits, axis=1)[:, -k:] # Get indices of top k predictions
    top_k_accuracy = np.mean([labels[i] in top_k_predictions[i] for i in range(len(labels))])

    # Combine results
    return {
        "accuracy": accuracy_result["accuracy"],
        "f1": f1_result["f1"],
        "top_5_accuracy": top_k_accuracy
    }

In [31]:
def conditional_train_model(model_instance, trainer_instance, training_args_instance, model_name_for_log):
    checkpoint_dir = training_args_instance.output_dir
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check for the existence of a model checkpoint file
    model_path = os.path.join(checkpoint_dir, "pytorch_model.bin")

    if os.path.exists(model_path) and not FORCE_FINE_TUNING:
        print(f"Checkpoint found for {model_name_for_log} at {checkpoint_dir}. Skipping training.")
        # Trainer automatically loads the best model if load_best_model_at_end=True
        # and a checkpoint exists in output_dir before evaluation/prediction.
        # So we just return 0 for train_runtime as no new training occurred.
        return 0.0
    else:
        if os.path.exists(checkpoint_dir) and FORCE_FINE_TUNING:
            print(f"Checkpoint found for {model_name_for_log}, but FORCE_FINE_TUNING is True. Re-training.")
        else:
            print(f"No checkpoint found for {model_name_for_log}. Starting fine-tuning.")

        train_result = trainer_instance.train()
        train_runtime = train_result.metrics["train_runtime"]
        print(f"Fine-tuning for {model_name_for_log} complete.")
        return train_runtime

### Freezing Backbone Layers

To fine-tune only the classification head, we need to freeze the parameters of the model's feature extractor (the backbone). This means we'll set `requires_grad=False` for all parameters except those belonging to the `model.head` module.

In [32]:
print(model)

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification head (typically named 'head' in timm models)
# You might need to inspect the model architecture (e.g., print(model)) if it's named differently
for param in model.classifier.parameters():
    param.requires_grad = True

print("Model parameters frozen. Only classification head will be fine-tuned.")

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [33]:
import evaluate
accuracy = evaluate.load("accuracy")

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_head_only", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ViT Head-Only fine-tuning...")
# Use the conditional training function
vit_head_only_train_time = conditional_train_model(model, trainer, training_args, "ViT Head-Only Fine-tune")
print("ViT Head-Only fine-tuning complete.")

Starting ViT Head-Only fine-tuning...
No checkpoint found for ViT Head-Only Fine-tune. Starting fine-tuning.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Top 5 Accuracy
1,4.537446,4.513514,0.106000,0.099083,0.310000
2,4.411290,4.403366,0.331000,0.322492,0.592000
3,4.327645,4.321104,0.451000,0.443183,0.723000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Fine-tuning the Whole Model (All Layers Unfrozen)

As requested, this section will fine-tune the entire Vision Transformer model (all layers, not just the classification head) on the Food101 dataset for 10 epochs.

In [ ]:
# Re-initialize the model to ensure no layers are frozen from previous steps
model_full_finetune = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Explicitly ensure all parameters require gradients for full fine-tuning
for param in model_full_finetune.parameters():
    param.requires_grad = True

print("Model re-initialized. All parameters are unfrozen and will be trained.")
print(model_full_finetune)

In [ ]:
training_args_full = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_full_finetune", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_full = Trainer(
    model=model_full_finetune,
    args=training_args_full,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting full model fine-tuning...")
# Use the conditional training function
vit_full_train_time = conditional_train_model(model_full_finetune, trainer_full, training_args_full, "ViT Full Fine-tune")
print("Full model fine-tuning complete.")

## LoRA Fine-tuning

This section demonstrates fine-tuning the Vision Transformer model using Low-Rank Adaptation (LoRA) to reduce computational cost and memory footprint during training.

In [ ]:
!pip install -q peft
!pip install --upgrade -q torchao

In [ ]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config_dict = {
    "r": 16,  # LoRA attention dimension
    "lora_alpha": 32,  # Alpha parameter for LoRA scaling
    "target_modules": ["query", "value"], # Target modules for LoRA. For ViT, 'query' and 'value' are common.
    "lora_dropout": 0.1,  # Dropout probability for LoRA layers
    "bias": "none",  # Bias type for LoRA layers
    "task_type": "CAUSAL_LM" # Task type. Set to 'CAUSAL_LM' for now, will adjust if needed.
}

# Re-initialize the base model
model_lora_base = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Adjust task type if necessary for image classification
# The task type 'CAUSAL_LM' is often used for text, for image classification, a different task_type might be more appropriate
# For image classification, there isn't a direct PEFT task_type, but we'll adapt.
# Let's set it to 'SEQ_CLS' (sequence classification) which is a common fallback for classification tasks if no specific image task type exists.
lora_config_dict['task_type'] = 'SEQ_CLS'
lora_config = LoraConfig(**lora_config_dict)

# Wrap the base model with LoRA
model_lora = get_peft_model(model_lora_base, lora_config)

print("LoRA model created:")
model_lora.print_trainable_parameters()
print(model_lora)

In [ ]:
# Set up TrainingArguments for LoRA fine-tuning
training_args_lora = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_lora", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5, # Start with a few epochs for LoRA
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_lora_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
# Use the conditional training function
vit_lora_train_time = conditional_train_model(model_lora, trainer_lora, training_args_lora, "ViT LoRA Fine-tune")
print("LoRA fine-tuning complete.")

# Task
The goal is to compare different fine-tuning strategies (head-only, full model, and LoRA) for a Vision Transformer (ViT) model on the Food101 dataset. The comparison will be based on performance metrics (accuracy and F1-score) and the number of trainable parameters. Additionally, the models will be evaluated on a separate CIFAR-10 dataset to assess their generalization capabilities.

## Install Additional Libraries

### Subtask:
Install necessary libraries such as `scikit-learn` for F1 score and confusion matrix, `matplotlib` and `seaborn` for plotting, and `accelerate` for optimized training.


In [ ]:
!pip install -q scikit-learn

In [ ]:
!pip install -q matplotlib seaborn accelerate

## Define Parameter Counting Utility

### Subtask:
Create a Python function to count and display the number of trainable parameters in a given PyTorch model. This function will be reused at various stages to track model complexity.


**Reasoning**:
The subtask requires defining a function to count trainable parameters. This code block implements that function by iterating through model parameters and summing up the elements of those that require gradients, then printing the result in millions.



In [ ]:
def count_parameters(model):
    """
    Counts and displays the number of trainable parameters in a PyTorch model.
    """
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Number of trainable parameters: {num_params_trainable:,} ({num_params_trainable / 1e6:.2f} million)")
    print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f} million)")


## Define Reusable Evaluation and Plotting Function

### Subtask:
Create a function that takes a trained model, a dataset, and label mappings as input, computes predictions, calculates accuracy and F1-score, and generates a confusion matrix. This function will be used to evaluate all fine-tuned models on both Food101 and CIFAR-10 datasets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
from tqdm.auto import tqdm
import numpy as np # Ensure numpy is imported for argmax and nan_to_num

def evaluate_and_plot(model, trainer, dataset, id2label_mapping, dataset_name, num_labels, model_display_name, training_time=None, normalize_cm=False, device='cuda'):
    print(f"\n--- Evaluating on {dataset_name} --- ")

    # Make predictions
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    labels = predictions.label_ids

    # Get predicted labels
    predicted_labels = np.argmax(logits, axis=1)

    # Print unique labels to verify distribution
    print(f"Unique true labels in {dataset_name}: {np.unique(labels)}")
    print(f"Unique predicted labels in {dataset_name}: {np.unique(predicted_labels)}")

    # Compute metrics using the shared compute_metrics function
    metrics = compute_metrics(predictions)
    print(f"Accuracy on {dataset_name}: {metrics['accuracy']:.4f}")
    print(f"F1-score (weighted) on {dataset_name}: {metrics['f1']:.4f}")
    print(f"Top-5 Accuracy on {dataset_name}: {metrics['top_5_accuracy']:.4f}")

    if training_time is not None:
        print(f"Training time for {model_display_name}: {training_time:.2f} seconds")

    # Generate Confusion Matrix
    cm = confusion_matrix(labels, predicted_labels)

    if normalize_cm:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm = np.nan_to_num(cm) # Handle cases where a row sums to zero (no true instances of a class)

    # Plot Confusion Matrix
    plt.figure(figsize=(12, 10))

    # Conditional plotting for readability with many classes
    if num_labels > 20:
        sns.heatmap(cm, cmap='Blues',
                    xticklabels=False, yticklabels=False, # Disable tick labels for clarity
                    cbar=True) # Keep color bar
        plt.title(f'Confusion Matrix for {model_display_name} on {dataset_name} (Labels Omitted)')
    else:
        sns.heatmap(cm, annot=True, fmt='.2f' if normalize_cm else 'g', cmap='Blues',
                    xticklabels=list(id2label_mapping.values()),
                    yticklabels=list(id2label_mapping.values()))
        plt.title(f'Confusion Matrix for {model_display_name} on {dataset_name}')

    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.tight_layout() # Adjust layout to prevent labels from being cut off
    plt.show()

    return metrics

## Evaluate Head-Only Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the head-only fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Head-Only Fine-tuning:")
count_parameters(model)

# Evaluate the head-only fine-tuned model on Food101 test set
head_only_metrics_food101 = evaluate_and_plot(
    model=model,
    trainer=trainer,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Head-Only)",
    num_labels=len(id2label),
    model_display_name="ViT Head-Only Fine-tune",
    training_time=vit_head_only_train_time
)

## Evaluate Full Model Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the full model fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Full Model Fine-tuning:")
count_parameters(model_full_finetune)

# Evaluate the full model fine-tuned model on Food101 test set
full_finetune_metrics_food101 = evaluate_and_plot(
    model=model_full_finetune,
    trainer=trainer_full,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Full Fine-tune)",
    num_labels=len(id2label),
    model_display_name="ViT Full Fine-tune",
    training_time=vit_full_train_time
)

## Evaluate LoRA Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the LoRA fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for LoRA Fine-tuning:")
model_lora.print_trainable_parameters() # LoRA models have their own method for this

# Evaluate the LoRA fine-tuned model on Food101 test set
lora_metrics_food101 = evaluate_and_plot(
    model=model_lora,
    trainer=trainer_lora,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (LoRA)",
    num_labels=len(id2label),
    model_display_name="ViT LoRA Fine-tune",
    training_time=vit_lora_train_time
)

## Fine-tuning Last 3 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 3 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model to ensure all layers are frozen initially
model_last_3_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers first
for param in model_last_3_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_3_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 3 encoder layers
num_encoder_layers = len(model_last_3_blocks.vit.encoder.layer)
for i in range(num_encoder_layers - 3, num_encoder_layers):
    for param in model_last_3_blocks.vit.encoder.layer[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 3 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_3_blocks)

### Train the Model (Last 3 Blocks + Head)

In [ ]:
training_args_last_3_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/my_awesome_food_model_last_3_blocks", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_last_3_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_3_blocks = Trainer(
    model=model_last_3_blocks,
    args=training_args_last_3_blocks,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 3 blocks + head...")
# Use the conditional training function
train_result_last_3_blocks = trainer_last_3_blocks.train()
vit_last_3_blocks_train_time = conditional_train_model(model_last_3_blocks, trainer_last_3_blocks, training_args_last_3_blocks, "ViT Last 3 Blocks + Head Fine-tune")
print("Fine-tuning for last 3 blocks + head complete.")

### Evaluate Model (Last 3 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 3 Blocks + Head Fine-tuning:")
count_parameters(model_last_3_blocks)

last_3_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_3_blocks,
    trainer=trainer_last_3_blocks,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 3 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 3 Blocks + Head Fine-tune",
    training_time=vit_last_3_blocks_train_time
)

## Fine-tuning Last 5 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 5 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model to ensure all layers are frozen initially
model_last_5_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers first
for param in model_last_5_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_5_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 5 encoder layers
num_encoder_layers = len(model_last_5_blocks.vit.encoder.layer)
for i in range(num_encoder_layers - 5, num_encoder_layers):
    for param in model_last_5_blocks.vit.encoder.layer[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 5 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_5_blocks)

### Train the Model (Last 5 Blocks + Head)

In [ ]:
training_args_last_5_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/my_awesome_food_model_last_5_blocks", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_last_5_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_5_blocks = Trainer(
    model=model_last_5_blocks,
    args=training_args_last_5_blocks,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 5 blocks + head...")
# Use the conditional training function
vit_last_5_blocks_train_time = conditional_train_model(model_last_5_blocks, trainer_last_5_blocks, training_args_last_5_blocks, "ViT Last 5 Blocks + Head Fine-tune")
print("Fine-tuning for last 5 blocks + head complete.")

### Evaluate Model (Last 5 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 5 Blocks + Head Fine-tuning:")
count_parameters(model_last_5_blocks)

last_5_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_5_blocks,
    trainer=trainer_last_5_blocks,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 5 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 5 Blocks + Head Fine-tune",
    training_time=vit_last_5_blocks_train_time
)

## Summary of Fine-tuning Experiment Results on Food101

In [ ]:
import pandas as pd

results_data = [
    {
        "Strategy": "ViT Head-Only Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args.num_train_epochs,
        "Learning Rate": training_args.learning_rate,
        "Training Time (seconds)": vit_head_only_train_time,
        "Accuracy": head_only_metrics_food101['accuracy'],
        "F1-score (weighted)": head_only_metrics_food101['f1'],
        "Top-5 Accuracy": head_only_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Full Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_full_finetune.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_full.num_train_epochs,
        "Learning Rate": training_args_full.learning_rate,
        "Training Time (seconds)": vit_full_train_time,
        "Accuracy": full_finetune_metrics_food101['accuracy'],
        "F1-score (weighted)": full_finetune_metrics_food101['f1'],
        "Top-5 Accuracy": full_finetune_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT LoRA Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_lora.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_lora.num_train_epochs,
        "Learning Rate": training_args_lora.learning_rate,
        "Training Time (seconds)": vit_lora_train_time,
        "Accuracy": lora_metrics_food101['accuracy'],
        "F1-score (weighted)": lora_metrics_food101['f1'],
        "Top-5 Accuracy": lora_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Last 3 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_3_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_3_blocks.num_train_epochs,
        "Learning Rate": training_args_last_3_blocks.learning_rate,
        "Training Time (seconds)": vit_last_3_blocks_train_time,
        "Accuracy": last_3_blocks_metrics_food101['accuracy'],
        "F1-score (weighted)": last_3_blocks_metrics_food101['f1'],
        "Top-5 Accuracy": last_3_blocks_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Last 5 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_5_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_5_blocks.num_train_epochs,
        "Learning Rate": training_args_last_5_blocks.learning_rate,
        "Training Time (seconds)": vit_last_5_blocks_train_time,
        "Accuracy": last_5_blocks_metrics_food101['accuracy'],
        "F1-score (weighted)": last_5_blocks_metrics_food101['f1'],
        "Top-5 Accuracy": last_5_blocks_metrics_food101['top_5_accuracy'],
    },
]

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(results_df)